##**Multi-Airport System**

###📌 **Overview**
This notebook solves a high-density airspace conflict where two airports—a Major Hub (ZBAA) and a Regional Node (ZBTJ)—share a single departure corridor. When a sudden disruption reduces corridor capacity by 50%, the model must decide how to re-allocate slots.

###⚙️ **The Core Logic**
We use Mixed-Integer Linear Programming (MILP) to navigate the trade-off between global system efficiency and stakeholder equity.

###🚀 **Quick Start: The 3 Scenarios**
The model is designed to run in three distinct modes. You can toggle the logic in the code to see the following results:

#### ✈ **Scenario 1 (Infeasible)**
Attempts to keep all flights strictly "On Time" (IATA < 15 min delay) under 50% capacity.
- On block 3, set max_allowed_delay = 3 (3 slots)
- On block 5, set the fairness_threshold = 150 (150 minutes)

#### ✈ **Scenario 2 (Efficiency Only)**
Minimizes total system delay but concentrates the entire burden on the
Regional airport. Effective on paper, unsustainable in reality.
- On block 3, set max_allowed_delay = 4

#### ✈ **Scenario 3 (The Fairness Balance)**
Introduces a Fairness Threshold to minimize delay disparity. Total delay remains the same as Scenario 2, but the burden is distributed equitably.
- On block 5, set the fairness_threshold = 15

In [56]:
## 1. Setup: The 45-minute peak window and Flight Database

from pulp import *

flights = [
    # Format: (flight_id, airport, requested_slot, flight_type)
    ('CCA-1501', 'Hub (ZBAA)', '08:00', 'Domestic'),
    ('CHH-7181', 'Hub (ZBAA)', '08:05', 'Domestic'),
    ('CKK-221',  'Hub (ZBAA)', '08:10', 'International'),
    ('CES-5106', 'Hub (ZBAA)', '08:15', 'Domestic'),
    ('CCA-981',  'Hub (ZBAA)', '08:25', 'International'),
    ('CSN-3104', 'Hub (ZBAA)', '08:35', 'Domestic'),
    ('CCA-1305', 'Hub (ZBAA)', '08:40', 'Domestic'),
    ('CHH-7602', 'Hub (ZBAA)', '08:45', 'Domestic'),
    ('GCR-6651', 'Regional (ZBTJ)', '08:00', 'Domestic'),
    ('OKN-2831', 'Regional (ZBTJ)', '08:05', 'Domestic'),
    ('CQH-8822', 'Regional (ZBTJ)', '08:20', 'Domestic'),
    ('XIA-8105', 'Regional (ZBTJ)', '08:30', 'Domestic'),
    ('GCR-6602', 'Regional (ZBTJ)', '08:35', 'Domestic'),
    ('OKN-2855', 'Regional (ZBTJ)', '08:40', 'Domestic'),
    ]

## Time intervals are 5 minutes each
slots = ["08:00", "08:05", "08:10", "08:15", "08:20", "08:25", "08:30", "08:35",
         "08:40", "08:45", "08:50", "08:55", "09:00", "09:05", "09:10"]

In [57]:
## 2. Initialize the Optimization Model

## We use LpMinimize because our goal is to reduce total system delay
model = LpProblem("Airport_Slot_Fairness_Model", LpMinimize)

In [58]:
## 3. Decision Variables: Binary matrix (1 if flight f is assigned to slot s, 0 otherwise)

# We use x_assign to clearly denote the decision of "assigning" a resource
x_assign = LpVariable.dicts("x_assign",
                            [(f[0], s) for f in flights for s in slots],
                            cat='Binary')

# 3A. Capacity Constraint: Ensuring a maximum of one flight per slot (Shared Tunnel)
for s in slots:
    model += lpSum(x_assign[f[0], s] for f in flights) <= 1

# 3B. Assignment Constraint: Every flight in the database must be dispatched
for f in flights:
    model += lpSum(x_assign[f[0], s] for s in slots) == 1

# 3C. NO-EARLY-DEPARTURE: Flights cannot depart before their requested time
# This is a critical physical constraint in airport operations
for f in flights:
    # Consistency Check: We unpack using the same naming convention as the flight database
    flight_id, airport, requested_slot, flight_type = f
    idx_req = slots.index(requested_slot)

    # Forbid all slots occurring before the requested time
    forbidden_slots = slots[:idx_req]
    for s_forbid in forbidden_slots:
        # We force the assignment to zero for any slot prior to the request
        model += x_assign[flight_id, s_forbid] == 0

# 3D. MAX ALLOWED DELAY: A control variable to manage system congestion
# This prevents excessive queuing and defines the feasible search space
max_allowed_delay = 3
for f in flights:
    # Unpacking again for clarity within this specific loop
    flight_id, airport, requested_slot, flight_type = f
    target_idx = slots.index(requested_slot)

    for s in slots:
        if slots.index(s) > target_idx + max_allowed_delay:
            model += x_assign[flight_id, s] == 0

In [59]:
## 4. Total Delay Calculations (Auxiliary Variables)

# 4A. We define these variables to monitor and control airport-specific performance
total_delay_hub = LpVariable("Total_Delay_Hub", lowBound=0)
total_delay_reg = LpVariable("Total_Delay_Regional", lowBound=0)

# Linking Constraints: Connecting these variables to the actual flight assignments

# 4B. Calculate total delay for the Hub (ZBAA)
model += total_delay_hub == lpSum((slots.index(s) - slots.index(f[2])) * 5 * x_assign[f[0], s]
                                  for f in flights for s in slots if f[1] == 'Hub (ZBAA)')

# 4C. Calculate total delay for the Regional Airport (ZBTJ)
model += total_delay_reg == lpSum((slots.index(s) - slots.index(f[2])) * 5 * x_assign[f[0], s]
                                  for f in flights for s in slots if f[1] == 'Regional (ZBTJ)')

In [60]:
## 5. Fairness Threshold: Balancing Airport Performance

fairness_threshold = 150  # Maximum allowed difference in minutes

# Linearizing the absolute value of the difference (The "Sandwich" approach)
model += total_delay_hub - total_delay_reg <= fairness_threshold
model += total_delay_reg - total_delay_hub <= fairness_threshold

In [61]:
## 6. Objective Function: Minimizing Global System Delay

# We aim to minimize the sum of accumulated delays across all airports.
model += total_delay_hub + total_delay_reg

In [62]:
# 7. Model Execution and Status Check
# We call the CBC solver. msg=0 hides the technical solver logs.
model.solve(PULP_CBC_CMD(msg=0))

print(f"Optimization Status: {LpStatus[model.status]}")

# Security check: If the model is not Optimal, we alert the user
if model.status != 1:  # 1 is the code for 'Optimal'
    print("Warning: The model could not find an optimal solution. Please check for conflicting constraints.")

Optimization Status: Optimal


In [63]:
# 8. Individual Flight Assignment Report
# We print the final slot for each flight and its specific delay in minutes.

print(f"Final Model Status: {LpStatus[model.status]}")
print("-" * 50)

for f in flights:
    # Unpacking for clarity and consistency with previous blocks
    flight_id, airport, requested_slot, flight_type = f
    assigned = False

    for s in slots:
        # We check the value of the binary decision variable
        if value(x_assign[flight_id, s]) == 1:
            # Calculate the delay in minutes (each slot = 5 minutes)
            delay_minutes = (slots.index(s) - slots.index(requested_slot)) * 5
            print(f"Flight {flight_id} ({airport}) -> Assigned: {s} | Delay: {delay_minutes} min")
            assigned = True
            break

    if not assigned:
        # This handles cases where the solver couldn't find a valid slot (Infeasibility)
        print(f"Flight {flight_id} ({airport}) -> ERROR: Not assigned (Check constraints)")

Final Model Status: Optimal
--------------------------------------------------
Flight CCA-1501 (Hub (ZBAA)) -> Assigned: 08:00 | Delay: 0 min
Flight CHH-7181 (Hub (ZBAA)) -> Assigned: 08:05 | Delay: 0 min
Flight CKK-221 (Hub (ZBAA)) -> Assigned: 08:10 | Delay: 0 min
Flight CES-5106 (Hub (ZBAA)) -> Assigned: 08:15 | Delay: 0 min
Flight CCA-981 (Hub (ZBAA)) -> Assigned: 08:30 | Delay: 5 min
Flight CSN-3104 (Hub (ZBAA)) -> Assigned: 08:35 | Delay: 0 min
Flight CCA-1305 (Hub (ZBAA)) -> Assigned: 08:45 | Delay: 5 min
Flight CHH-7602 (Hub (ZBAA)) -> Assigned: 09:05 | Delay: 20 min
Flight GCR-6651 (Regional (ZBTJ)) -> Assigned: 08:20 | Delay: 20 min
Flight OKN-2831 (Regional (ZBTJ)) -> Assigned: 08:25 | Delay: 20 min
Flight CQH-8822 (Regional (ZBTJ)) -> Assigned: 08:40 | Delay: 20 min
Flight XIA-8105 (Regional (ZBTJ)) -> Assigned: 08:50 | Delay: 20 min
Flight GCR-6602 (Regional (ZBTJ)) -> Assigned: 08:55 | Delay: 20 min
Flight OKN-2855 (Regional (ZBTJ)) -> Assigned: 09:00 | Delay: 20 min


In [64]:
# 9. KPI Analysis: Business Impact Summary
# We initialize our performance counters to verify the fairness results
analysis_results = {
    "By Airport": {"Hub (ZBAA)": 0, "Regional (ZBTJ)": 0},
    "By Type": {"International": 0, "Domestic": 0}
}

for f in flights:
    # Consistency Unpacking: (flight_id, airport, requested_slot, flight_type)
    flight_id, airport, requested_slot, flight_type = f

    for s in slots:
        # Check the decision variable from our assignment matrix
        if value(x_assign[flight_id, s]) == 1:
            delay_minutes = (slots.index(s) - slots.index(requested_slot)) * 5

            # Aggregate results for strategic KPIs
            analysis_results["By Airport"][airport] += delay_minutes
            analysis_results["By Type"][flight_type] += delay_minutes
            break # Optimized: Move to next flight once found

# --- Performance Dashboard ---
print("\n" + "="*40)
print(" STRATEGIC KPI REPORT ")
print("="*40)

# A. Airport Fairness Verification
hub_delay = analysis_results['By Airport']['Hub (ZBAA)']
reg_delay = analysis_results['By Airport']['Regional (ZBTJ)']
gap = abs(hub_delay - reg_delay)

print(f"Total Delay Hub (ZBAA):      {hub_delay} min")
print(f"Total Delay Regional (ZBTJ): {reg_delay} min")
print(f"CURRENT FAIRNESS GAP:        {gap} min")
print("-" * 40)

# B. Operational Impact by Flight Type
print(f"Domestic Flights Delay:      {analysis_results['By Type']['Domestic']} min")
print(f"International Flights Delay: {analysis_results['By Type']['International']} min")
print("="*40)


 STRATEGIC KPI REPORT 
Total Delay Hub (ZBAA):      30 min
Total Delay Regional (ZBTJ): 120 min
CURRENT FAIRNESS GAP:        90 min
----------------------------------------
Domestic Flights Delay:      145 min
International Flights Delay: 5 min
